In [ ]:
import pandas as pd

df = pd.read_csv("online_retail_II.csv")

print(df.dtypes)
df.head()

In [ ]:
has_dot_zero = df["Customer ID"].astype(str).str.contains(".0")

df[has_dot_zero].shape, df[df["Customer ID"].notna()].shape, df.shape

In [ ]:
import pandas as pd

dtype = {
    "Invoice": "str",
    "StockCode": "str",
    "Description": "str",
    "Quantity": "int",
    "InvoiceDate": "str",
    "Price": "float",
    "Customer ID": "str",
    "Country": "str",
}
df = pd.read_csv("online_retail_II.csv", dtype=dtype)
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
#df["Customer ID"] = df["Customer ID"].str.replace(".0", "")
df.columns = [
    "invoice", "stock_code", "description", "quantity",
    "invoice_date", "price", "customer_id", "country"
]
df.head()

In [ ]:
df.dtypes

In [ ]:
print(f"{df.shape=}")

for c in df.columns:
    print(f"{c}, {df[c].isna().sum()}")

In [ ]:
df.info()

In [ ]:
df["invoice"].describe()

In [ ]:
df["invoice"].nunique(), df["invoice_date"].nunique()

In [ ]:
df.describe(include="all")

# InvoiceNo で "C" を含むもの

In [ ]:
is_cancelled = df["invoice"].str.contains("C")
print(df[is_cancelled].shape)

df[is_cancelled].head()

In [ ]:
df.loc[(~is_cancelled) & (df["quantity"] < 0), ["invoice", "quantity", "price"]].head()

In [ ]:
df.loc[(~is_cancelled) & (df["price"] < 0), ["invoice", "quantity", "price"]]

In [ ]:
df.loc[is_cancelled | (df["price"] < 0) | (df["quantity"] < 0)].shape, df.shape

In [ ]:
df.loc[is_cancelled, "invoice"].unique()

# Quantity

In [ ]:
print(len(df[df["quantity"]<0]))
df[df["quantity"]<0].head()

# UnitPrice

In [ ]:
df["price"].describe()

In [ ]:
df[df["price"]<0]

In [ ]:
df[df["customer_id"].notna()].reset_index(drop=True).shape

In [ ]:
for c in df.columns:
    print(f"{c}, {df[c].nunique()}")

In [ ]:
check_cols = ["invoice", "invoice_date"]

check_ids = df["customer_id"].sample(n=20, random_state=42)
for cid in check_ids:
    tmp = df[df["customer_id"]==cid]
    nuni_no = tmp["invoice"].nunique()
    nuni_date = tmp["invoice_date"].nunique()
    print(f"{nuni_no=}, {nuni_date=}")
    if nuni_no != nuni_date:
        no_dates = set(tmp["invoice"].astype(str) + " + " + tmp["invoice_date"].astype(str))
        for no_date in sorted(no_dates):
            print(no_date)
    break

## StockCode, Descriptionの確認

In [ ]:
df["invoice"].nunique(), df["stock_code"].nunique()

In [ ]:
df["stock_code"].nunique(), df["description"].nunique()

In [ ]:
df.groupby("stock_code")["description"].nunique().value_counts()

In [ ]:
df["stock_code"].value_counts()

In [ ]:
dup_sk = df.groupby("stock_code")["description"].nunique()  # stock_code に対して、description が複数存在するものを抽出
dup_sk

In [ ]:
dup_sk = dup_sk.loc[dup_sk>1]  # description が 2 つ以上存在する stock_code を抽出
stock_code_with_multi_description = dup_sk.sort_values(ascending=False).index.tolist()  # 降順にしてリスト化
stock_code_with_multi_description[:5]

In [ ]:
stock_code = "85123A"
df.loc[df["stock_code"]==stock_code, "description"].unique()

In [ ]:
stock_code = "20725"
df.loc[df["stock_code"]==stock_code, "description"].unique()

In [ ]:
stock_code = "82486"
df.loc[df["stock_code"]==stock_code, "description"].unique()

## Quantityの確認

In [ ]:
df["quantity"].describe()

In [ ]:
df[df["quantity"]<1]

In [ ]:
df[df["quantity"]<1].head(20)

## Country

In [ ]:
agg_cols = ["invoice", "stock_code", "invoice_date", "country"]
tmp = df.groupby(["customer_id"]).nunique()[agg_cols]
tmp[tmp["country"]>1]

In [ ]:
df["country"].value_counts().head(10)

In [ ]:
tmp = df.groupby(["customer_id"]).nunique()[["country"]]
tmp[tmp["country"]>1].sort_values("country", ascending=False)

In [ ]:
customer_id_with_multi_countries = tmp[tmp["country"]>1].reset_index()["customer_id"].to_list()
for customer_id in customer_id_with_multi_countries:
    print(df.loc[df["customer_id"]==customer_id, "country"].value_counts())

## InvoiceDate

In [ ]:
tmp = df.groupby(["invoice"]).nunique()[["invoice_date"]]
tmp[tmp["invoice_date"]>1].sort_values("invoice_date", ascending=False).head()

In [ ]:
df.loc[df["invoice"]=="536591", ["invoice", "invoice_date"]].drop_duplicates()

In [ ]:
df["invoice_date"].min(), df["invoice_date"].max()

In [ ]:
df["invoice_date"].dt.strftime("%Y-%m-%d").value_counts()

In [ ]:
monthly_df = df["invoice_date"].dt.strftime("%Y-%m").value_counts().reset_index().sort_values(by="invoice_date").reset_index(drop=True)
monthly_df

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.bar(monthly_df['invoice_date'], monthly_df['count'])
plt.xlabel('invoice_date')
plt.ylabel('count')
plt.title('Count by invoice_date')
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

In [ ]:
df["invoice_month"] = df["invoice_date"].dt.strftime("%Y-%m")
df["purchase_amount"] = df["price"] * df["quantity"]
agg_cols = ["invoice", "stock_code", "customer_id"]

agg_dict = {
    "invoice": "nunique",
    "stock_code": "nunique",
    "customer_id": "nunique",
    "purchase_amount": ["sum", "mean"]
}
monthly_df = df.groupby("invoice_month").agg(agg_dict).reset_index()
monthly_df.columns = ['_'.join(col).strip() if col[1] else col[0] for col in monthly_df.columns]

for col in monthly_df.columns[1:]:
    plt.figure(figsize=(8, 3))
    plt.bar(monthly_df['invoice_month'], monthly_df[col])
    plt.xlabel('invoice_month')
    plt.ylabel(col)
    plt.xticks(rotation=90)
    plt.tight_layout()
    plt.savefig(f"figures/{col}.png", dpi=300, bbox_inches='tight')
    plt.show()